<a href="https://colab.research.google.com/github/zzpsah/eshikshakosh-automation/blob/main/Eshikshakosh_OTR_ORIGINAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎓 e-ShikshaKosh Bihar — Complete Student, OTR & Bank Verification Report
### Made with ❤️ by Eternal Student

---

If this tool saves your time and effort, consider supporting with Tea ☕ or Coffee ☕

**UPI:** `eternalstudent@cnrb`

---
### 📌 How to use:
1. **Step 1: Setup Environment** — Run once to install required packages.
2. **Step 2: Authentication & Login** — Enter your Portal **User ID** (School UDISE) and **Password**. Session scope is automatically locked to your school.
3. **Step 3: Download Complete Verification Report** — Automatically fetches all students, resolves **Mother's Name**, **Bank Account No**, **IFSC Code**, **Bank Name (Auto)**, **Account Holder Name**, **OTR Number**, **OTR Status**, and unmasked **Aadhaar Number**, then exports a clean Excel sheet ready for teacher validation.


In [ ]:
#@title ⚙️ Step 1: Setup Environment { display-mode: "form" }

!pip install -q requests pandas openpyxl tqdm playwright nest_asyncio
!playwright install chromium
!playwright install-deps chromium

import os, sys, json, time, re, logging, urllib3, asyncio, base64, ssl
import requests
import pandas as pd
import nest_asyncio
nest_asyncio.apply()

from pathlib import Path
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from openpyxl import Workbook, load_workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
try:
    ssl._create_default_https_context = ssl._create_unverified_context
except Exception:
    pass

print("Environment Ready.")

In [ ]:
#@title 🔐 Step 2: Authentication & Login { display-mode: "form" }
import json, base64, asyncio, re, time
import nest_asyncio
nest_asyncio.apply()

BASE_URL        = "https://eshikshakosh.bihar.gov.in:8443"
LIST_URL        = f"{BASE_URL}/student/viewStudentList"
DETAIL_URL      = f"{BASE_URL}/student/studentInfo"
AADHAAR_REF_URL = f"{BASE_URL}/student/getAadhaarFromRef"

HEADERS = {}
SESSION_CONTEXT = {}

SESSION = requests.Session()
SESSION.verify = False
retries = Retry(total=4, backoff_factor=0.6, status_forcelist=[429, 500, 502, 503, 504], allowed_methods=["POST", "GET"])
adapter = HTTPAdapter(max_retries=retries, pool_maxsize=30)
SESSION.mount("https://", adapter)
SESSION.mount("http://", adapter)

def decode_jwt(token_str):
    try:
        clean = token_str.replace("Bearer", "").strip()
        payload_b64 = clean.split(".")[1]
        payload_b64 += "=" * (-len(payload_b64) % 4)
        return json.loads(base64.urlsafe_b64decode(payload_b64).decode("utf-8"))
    except Exception:
        return {}

user_id = input("Enter Portal User ID (UDISE): ").strip()
try:
    from getpass import getpass
    password = getpass("Enter Portal Password: ").strip()
except Exception:
    password = ""
if not password:
    password = input("Enter Portal Password: ").strip()

async def run_playwright_login_async(uid, pwd):
    captured_dict = {}
    async def handle_response(response):
        if "auth/login" in response.url and response.status == 200:
            try:
                data = await response.json()
                if data.get("token"):
                    captured_dict["token"] = data["token"]
                    captured_dict["access_token"] = data.get("access_token", data["token"])
            except Exception:
                pass

    try:
        from playwright.async_api import async_playwright
        async with async_playwright() as p:
            browser = await p.chromium.launch(
                headless=True,
                args=["--no-sandbox", "--disable-setuid-sandbox", "--disable-dev-shm-usage", "--disable-gpu"]
            )
            context = await browser.new_context(ignore_https_errors=True)
            page = await context.new_page()
            page.on("response", handle_response)

            print("Connecting to portal...")
            await page.goto("https://eshikshakosh.bihar.gov.in", wait_until="domcontentloaded", timeout=45000)

            radio = await page.wait_for_selector("#radio2", timeout=25000)
            await radio.click()
            await page.wait_for_timeout(400)

            await page.wait_for_selector("input[formcontrolname='userId']", timeout=15000)
            await page.fill("input[formcontrolname='userId']", uid)
            await page.fill("input[formcontrolname='password']", pwd)

            await page.wait_for_selector(".capcha__head", timeout=12000)
            captcha_text = await page.inner_text(".capcha__head")
            m = re.search(r"(\d+)\s*([\+\-\*])\s*(\d+)", captcha_text)
            if m:
                n1, op, n2 = int(m.group(1)), m.group(2), int(m.group(3))
                ans = n1 + n2 if op == "+" else (n1 - n2 if op == "-" else n1 * n2)
            else:
                expr = captcha_text.split(":")[-1].strip()
                ans = int(eval(expr))

            await page.fill("input[placeholder='Enter User Captcha']", str(ans))
            await page.click("input.submit__btn")

            for _ in range(120):
                if captured_dict.get("token"):
                    break
                await asyncio.sleep(0.5)
            await browser.close()
    except Exception as e:
        print(f"Notice: {e}")
    return captured_dict

loop = asyncio.get_event_loop()
captured = loop.run_until_complete(run_playwright_login_async(user_id, password))

token = captured.get("token")
access_token = captured.get("access_token", token)

if token:
    HEADERS.update({
        "accept": "application/json, text/plain, */*",
        "content-type": "application/json",
        "origin": "https://eshikshakosh.bihar.gov.in",
        "referer": "https://eshikshakosh.bihar.gov.in/",
        "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "accesstoken": f"Bearer {access_token}",
        "authaccess": f"Bearer {access_token}",
        "authorization": f"Bearer {token}",
        "authtoken": f"Bearer {token}",
    })

    jwt_claims = decode_jwt(token)
    school_id = jwt_claims.get("sub")
    udise_code = jwt_claims.get("userId") or user_id

    SESSION_CONTEXT = {
        "user_id": user_id,
        "school_id": school_id,
        "udise_code": udise_code,
        "token": token,
        "access_token": access_token
    }
    print("Login Successful! Proceed to Step 3.")
else:
    raise RuntimeError("Login failed. Please verify your User ID and Password, and ensure Step 1 is executed.")

In [ ]:
#@title 📊 Step 3: Fetch Students & Download Clean Excel Report { display-mode: "form" }

if not SESSION_CONTEXT.get("token"):
    raise RuntimeError("Please run Step 2 first to authenticate.")

SESSION.verify = False
school_id = SESSION_CONTEXT.get("school_id")
udise_code = SESSION_CONTEXT.get("udise_code")

if not school_id:
    raise RuntimeError("Could not detect School ID from session. Please re-run Step 2.")

ac_year = input("Enter Academic Year [Press Enter for 2026-27]: ").strip() or "2026-27"
print(f"\n▶ Step 3.1: Fetching student roster for academic year {ac_year}...")

# 1. Fetch Student List from viewStudentList with pagination
all_students = []
offset = 0
limit = 500

while True:
    payload = {
        "offset": str(offset),
        "limit": str(limit),
        "searchSchoolId": school_id,
        "academicYear": ac_year,
    }
    try:
        res = SESSION.post(LIST_URL, headers=HEADERS, json=payload, timeout=35, verify=False)
        if res.status_code != 200:
            print(f"Error fetching student batch at offset {offset}: HTTP {res.status_code}")
            break
        res_json = res.json()
        batch = res_json.get("data", [])
        if not batch:
            break
        all_students.extend(batch)
        total_record = res_json.get("totalRecord", len(all_students))
        print(f"   Loaded {len(all_students)} of {total_record} students...")
        if len(all_students) >= total_record or len(batch) < limit:
            break
        offset += limit
    except Exception as e:
        print(f"Error fetching roster batch: {e}")
        break

if not all_students:
    raise RuntimeError(f"No student records found for academic year {ac_year}.")

school_name = all_students[0].get("schoolName") or f"School_{udise_code}"
print(f"   Total Students: {len(all_students)}")

# 2. Helper to validate real 12-digit Aadhaar
def is_valid_aadhaar(val):
    s = str(val or "").strip()
    return bool(re.match(r"^\d{12}$", s))

# 3. Robust student profile fetcher (Mother Name, Bank Account, IFSC, Acc Holder) with retries
print("\n▶ Step 3.2: Fetching complete profile details (Mother Name, Bank Details, Account)...")
student_details = {}

def fetch_student_detail_robust(enc_id, ac_year, max_retries=4):
    if not enc_id:
        return {}
    payload = {"encId": str(enc_id), "academicYear": ac_year}
    for attempt in range(max_retries):
        try:
            res = SESSION.post(DETAIL_URL, headers=HEADERS, json=payload, timeout=20, verify=False)
            if res.status_code == 200:
                res_json = res.json()
                data = res_json.get("data")
                if isinstance(data, dict) and data:
                    return data
            time.sleep(0.4 * (attempt + 1))
        except Exception:
            time.sleep(0.4 * (attempt + 1))
    return {}

def worker_fetch(s):
    s_code = str(s.get("studentCode") or "").strip()
    enc = s.get("studentId")
    data = fetch_student_detail_robust(enc, ac_year)
    return s_code, data

with ThreadPoolExecutor(max_workers=8) as pool:
    detail_results = list(pool.map(worker_fetch, all_students))
    for s_code, d in detail_results:
        if d:
            student_details[s_code] = d

# Sweep pass for any records that had temporary network/server lock
missing_students = [s for s in all_students if not student_details.get(str(s.get("studentCode") or "").strip(), {}).get("motherName")]
if missing_students:
    print(f"   Verifying {len(missing_students)} remaining profile records...")
    for s in missing_students:
        s_code = str(s.get("studentCode") or "").strip()
        enc = s.get("studentId")
        if enc:
            d = fetch_student_detail_robust(enc, ac_year, max_retries=3)
            if d:
                student_details[s_code] = d
        time.sleep(0.05)

print(f"   Profile details retrieved for {len(student_details)} students.")

# 4. Resolve unmasked Aadhaar numbers
print("\n▶ Step 3.3: Resolving full unmasked Aadhaar numbers...")

def fetch_aadhaar_robust(ref_no, max_retries=3):
    clean_ref = str(ref_no).strip()
    if not clean_ref:
        return clean_ref, None
    for attempt in range(max_retries):
        try:
            res = SESSION.post(AADHAAR_REF_URL, headers=HEADERS, json={"referenceNo": clean_ref}, timeout=15, verify=False)
            if res.status_code == 200:
                res_json = res.json()
                data = res_json.get("data")
                if data and is_valid_aadhaar(data):
                    return clean_ref, str(data).strip()
            time.sleep(0.4 * (attempt + 1))
        except Exception:
            time.sleep(0.4 * (attempt + 1))
    return clean_ref, None

# Collect reference keys for ANY student whose Aadhaar is not a valid 12-digit number
ref_keys_to_fetch = set()
for s in all_students:
    s_code = str(s.get("studentCode") or "").strip()
    dt = student_details.get(s_code, {})

    # If neither s nor dt has a valid 12-digit Aadhaar
    if not is_valid_aadhaar(s.get("aadhaar")) and not is_valid_aadhaar(dt.get("aadhaar")):
        for k in [s.get("referenceKey"), dt.get("referenceKey"), s.get("maskedReferenceKey"), dt.get("maskedReferenceKey")]:
            if k:
                digits_only = re.sub(r"\D", "", str(k))
                if len(digits_only) >= 6:
                    ref_keys_to_fetch.add(digits_only)

resolved_aadhaars = {}
if ref_keys_to_fetch:
    print(f"   Resolving {len(ref_keys_to_fetch)} Aadhaar numbers via reference keys...")
    with ThreadPoolExecutor(max_workers=5) as pool:
        futures = [pool.submit(fetch_aadhaar_robust, ref) for ref in ref_keys_to_fetch]
        for fut in as_completed(futures):
            ref_k, aadhar_val = fut.result()
            if aadhar_val:
                resolved_aadhaars[ref_k] = aadhar_val
    print(f"   Unmasked Aadhaar resolved for {len(resolved_aadhaars)} students.")

# 5. Bank Name Auto-Determination from IFSC
print("\n▶ Step 3.4: Auto-determining Bank Names from IFSC codes...")
IFSC_PREFIX_MAP = {
    "SBIN": "State Bank of India",
    "PUNB": "Punjab National Bank",
    "CBIN": "Central Bank of India",
    "UBIN": "Union Bank of India",
    "BKID": "Bank of India",
    "BARB": "Bank of Baroda",
    "CNRB": "Canara Bank",
    "UCOB": "UCO Bank",
    "IDIB": "Indian Bank",
    "IOBA": "Indian Overseas Bank",
    "IPOS": "India Post Payments Bank",
    "HDFC": "HDFC Bank",
    "ICIC": "ICICI Bank",
    "UTIB": "Axis Bank",
    "KKBK": "Kotak Mahindra Bank",
    "AIRP": "Airtel Payments Bank",
    "PYTM": "Paytm Payments Bank",
    "FINO": "Fino Payments Bank",
    "BDBL": "Bandhan Bank",
    "IBKL": "IDBI Bank",
    "YESB": "Yes Bank",
    "IDFB": "IDFC First Bank",
    "UBGB": "Uttar Bihar Gramin Bank",
    "DBGB": "Dakshin Bihar Gramin Bank",
}

BANK_NAME_CACHE = {
    "CBIN0R10001": "Uttar Bihar Gramin Bank",
    "IPOS0000001": "India Post Payments Bank",
}

def resolve_bank_name(ifsc_code):
    if not ifsc_code:
        return ""
    code = str(ifsc_code).strip().upper()
    if len(code) < 4:
        return ""
    if code in BANK_NAME_CACHE:
        return BANK_NAME_CACHE[code]
    try:
        r = requests.get(f"https://ifsc.razorpay.com/{code}", timeout=3)
        if r.status_code == 200:
            b_name = r.json().get("BANK", "")
            if b_name:
                BANK_NAME_CACHE[code] = b_name
                return b_name
    except Exception:
        pass
    prefix = code[:4]
    name = IFSC_PREFIX_MAP.get(prefix, prefix)
    BANK_NAME_CACHE[code] = name
    return name

unique_ifscs = set()
for s in all_students:
    s_code = str(s.get("studentCode") or "").strip()
    dt = student_details.get(s_code, {})
    ifsc = dt.get("IFSC") or s.get("IFSC")
    if ifsc:
        unique_ifscs.add(str(ifsc).strip().upper())

for code in unique_ifscs:
    resolve_bank_name(code)

# 6. Build Final Table Records
STREAM_MAP = {1: "Arts", 2: "Science", 3: "Commerce", 0: "N/A"}
records = []

for idx, s in enumerate(all_students, 1):
    s_code = str(s.get("studentCode") or "").strip()
    dt = student_details.get(s_code, {})

    # Mother Name
    mother_name = str(dt.get("motherName") or s.get("motherName") or "").strip()
    if mother_name in ["None", "null", "nan", "N/A"]:
        mother_name = ""

    # Father Name
    father_name = str(dt.get("fatherName") or s.get("fatherName") or "").strip()
    if father_name in ["None", "null", "nan"]:
        father_name = ""

    # Bank Account & IFSC
    bank_acc = str(dt.get("bankAccNo") or s.get("bankAccNo") or "").strip()
    if bank_acc in ["None", "null", "0", "nan"]:
        bank_acc = ""
    ifsc_code = str(dt.get("IFSC") or s.get("IFSC") or "").strip().upper()
    if ifsc_code in ["None", "null", "nan"]:
        ifsc_code = ""

    bank_name = resolve_bank_name(ifsc_code) if ifsc_code else ""
    acc_holder = str(dt.get("accHolderName") or s.get("accHolderName") or "").strip()
    if acc_holder in ["None", "null", "nan"]:
        acc_holder = ""

    # Aadhaar Number resolution: valid 12-digit direct -> resolved from ref -> masked -> blank
    final_aadhaar = ""
    for candidate in [dt.get("aadhaar"), s.get("aadhaar")]:
        if is_valid_aadhaar(candidate):
            final_aadhaar = str(candidate).strip()
            break

    if not final_aadhaar:
        for rk in [s.get("referenceKey"), dt.get("referenceKey"), s.get("maskedReferenceKey"), dt.get("maskedReferenceKey")]:
            digits = re.sub(r"\D", "", str(rk or ""))
            if digits in resolved_aadhaars and is_valid_aadhaar(resolved_aadhaars[digits]):
                final_aadhaar = resolved_aadhaars[digits]
                break

    if not final_aadhaar:
        for m in [s.get("maskedAadhaar"), dt.get("maskedAadhaar")]:
            m_str = str(m or "").strip()
            if m_str and m_str not in ["--", "None", "null", "nan"]:
                final_aadhaar = m_str
                break

    # OTR Details
    otr_no = str(s.get("otrNo") or dt.get("otrNo") or "").strip()
    if otr_no in ["None", "null", "0", "nan"]:
        otr_no = ""
    otr_flag = s.get("otrFlag") or dt.get("otrFlag")
    otr_status = "Registered" if (otr_no or str(otr_flag) == "1") else "Pending"

    # Stream
    st_val = s.get("stream") if s.get("stream") is not None else dt.get("stream")
    stream_text = STREAM_MAP.get(st_val, str(st_val) if st_val is not None else "N/A")
    if str(s.get("class", "")) not in ["11", "12"]:
        stream_text = "N/A"

    # CWSN & Profile Status
    cwsn_val = s.get("cwsn") or dt.get("cwsn")
    cwsn_text = "Yes" if str(cwsn_val) in ["1", "2"] else "No"
    prof_status = s.get("stdProfileUpdatedStatus") or dt.get("stdProfileUpdatedStatus")
    profile_updated = "Yes" if str(prof_status) == "1" else "No"

    records.append({
        "S.No.": idx,
        "Student Code": s_code,
        "Student Name": str(s.get("studentName") or dt.get("studentName") or "").strip(),
        "Father's Name": father_name,
        "Mother's Name": mother_name,
        "Class": str(s.get("className") or dt.get("className") or (f"Class {s.get('class')}" if s.get('class') else "")).strip(),
        "Section": str(s.get("section") or dt.get("section") or "").strip(),
        "Roll No": str(s.get("rollNo") or dt.get("rollNo") or "").strip(),
        "OTR Number": otr_no,
        "OTR Status": otr_status,
        "Aadhaar Number": final_aadhaar,
        "Bank Account No": bank_acc,
        "IFSC Code": ifsc_code,
        "Bank Name": bank_name,
        "Account Holder Name": acc_holder,
        "Mobile Number": str(s.get("mobile") or dt.get("mobile") or "").strip(),
        "Date of Birth": str(s.get("dob") or dt.get("dob") or "").strip(),
        "Gender": str(s.get("genderName") or dt.get("genderName") or ("Male" if s.get("gender") == 1 else "Female" if s.get("gender") == 2 else "")).strip(),
        "Social Category": str(s.get("socialCategory") or dt.get("socialCategory") or "").strip(),
        "Admission No": str(s.get("admissionNo") or dt.get("admissionNo") or "").strip(),
        "Admission Date": str(s.get("admissionDate") or dt.get("admissionDate") or "").strip(),
        "Stream": stream_text,
        "CWSN": cwsn_text,
        "Profile Updated": profile_updated,
        "School Name": school_name,
        "UDISE Code": str(udise_code).strip(),
        "Academic Year": ac_year,
    })

df = pd.DataFrame(records)

# 7. Generate Clean, Professional Excel File for Teachers
out_filename = f"Student_OTR_Report_{udise_code}_{ac_year}.xlsx"

wb = Workbook()
ws = wb.active
ws.title = "Student OTR Roster"

columns = list(df.columns)
ws.append(columns)

header_fill = PatternFill(start_color="F2F4F7", end_color="F2F4F7", fill_type="solid")
header_font = Font(name="Calibri", size=10, bold=True, color="1F2937")
cell_font   = Font(name="Calibri", size=10, color="111827")
thin_border = Border(
    left=Side(style="thin", color="D1D5DB"),
    right=Side(style="thin", color="D1D5DB"),
    top=Side(style="thin", color="D1D5DB"),
    bottom=Side(style="thin", color="D1D5DB")
)

align_center = Alignment(horizontal="center", vertical="center")
align_left   = Alignment(horizontal="left", vertical="center")

ws.row_dimensions[1].height = 25
for col_idx, cell in enumerate(ws[1], 1):
    cell.fill = header_fill
    cell.font = header_font
    cell.alignment = align_center
    cell.border = thin_border

TEXT_COLUMNS = [
    "Student Code", "Roll No", "OTR Number", "Aadhaar Number",
    "Bank Account No", "IFSC Code", "Mobile Number", "Admission No", "UDISE Code"
]
text_col_indices = [columns.index(c) + 1 for c in TEXT_COLUMNS if c in columns]

center_col_names = [
    "S.No.", "Class", "Section", "Roll No", "OTR Number", "OTR Status",
    "Aadhaar Number", "Bank Account No", "IFSC Code", "Mobile Number",
    "Date of Birth", "Gender", "Social Category", "Admission Date", "Stream",
    "CWSN", "Profile Updated", "UDISE Code", "Academic Year"
]
center_col_indices = [columns.index(c) + 1 for c in center_col_names if c in columns]

for row_idx, row_data in enumerate(records, 2):
    ws.row_dimensions[row_idx].height = 20
    row_values = [row_data[col] for col in columns]
    ws.append(row_values)
    for col_idx in range(1, len(columns) + 1):
        cell = ws.cell(row=row_idx, column=col_idx)
        cell.font = cell_font
        cell.border = thin_border

        if col_idx in text_col_indices:
            cell.number_format = "@"
            if cell.value is not None:
                cell.value = str(cell.value).strip()

        if col_idx in center_col_indices:
            cell.alignment = align_center
        else:
            cell.alignment = align_left

for col in ws.columns:
    col_letter = get_column_letter(col[0].column)
    max_len = max(len(str(cell.value or "")) for cell in col)
    ws.column_dimensions[col_letter].width = max(max_len + 4, 13)

ws.freeze_panes = "A2"
ws.auto_filter.ref = ws.dimensions

wb.save(out_filename)

try:
    from google.colab import files
    files.download(out_filename)
except Exception:
    pass

registered_count = sum(1 for r in records if r["OTR Status"] == "Registered")
pending_count    = len(records) - registered_count
aadhaar_count    = sum(1 for r in records if is_valid_aadhaar(r["Aadhaar Number"]))
bank_count       = sum(1 for r in records if r["Bank Account No"] != "")
mother_count     = sum(1 for r in records if r["Mother's Name"] != "")

print("\n" + "=" * 65)
print("Report Generated & Downloaded Successfully!")
print(f"File Saved: {out_filename}")
print(f"Summary:")
print(f"   • Total Students        : {len(records)}")
print(f"   • OTR Registered        : {registered_count}")
print(f"   • OTR Pending           : {pending_count}")
print(f"   • Unmasked 12-d Aadhaar : {aadhaar_count}")
print(f"   • Bank Accounts         : {bank_count}")
print(f"   • Mother Names          : {mother_count}")
print("=" * 65)